In [ ]:
import fastf1
from fastf1 import get_event_schedule, get_session
import pandas as pd
import os
import logging
import re
from pathlib import Path

# ========== CONFIG ==========
START_SEASON = 2018
END_SEASON = 2025
WET_LAP_THRESHOLD = 0.1
cache_dir = '.fastf1_cache'
output_dir = 'data/processed'
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

fastf1.Cache.enable_cache(cache_dir)

compound_map = {
    'SOFT': 'Soft', 'MEDIUM': 'Medium', 'HARD': 'Hard',
    'INTERMEDIATE': 'Intermediate', 'WET': 'Wet'
}

def slugify(text):
    """Basic slugify for safe filenames."""
    return re.sub(r'[^\w]+', '_', text.lower()).strip('_')

# ========== LOGGING ==========
logging.basicConfig(
    filename=f'fastf1_data_export_{START_SEASON}_{END_SEASON}.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print(f"\n{'='*70}")
print(f"🏁 F1 DATA EXTRACTION - USING YOUR WORKING APPROACH")
print(f"{'='*70}")
print(f"📥 Extracting: {START_SEASON}-{END_SEASON}")
print(f"⏱️  Estimated time: {(END_SEASON - START_SEASON + 1) * 1.5:.1f}-{(END_SEASON - START_SEASON + 1) * 2.5:.1f} hours\n")

# Process each season
for SEASON in range(START_SEASON, END_SEASON + 1):
    print(f"\n{'='*70}")
    print(f"🏁 Processing Season {SEASON}")
    print(f"{'='*70}\n")
    
    all_races = []
    
    try:
        # THIS IS YOUR WORKING APPROACH - Simple, no backend specification
        schedule = get_event_schedule(SEASON, include_testing=False)
        race_events = schedule.copy()
        
        print(f"✅ Found {len(race_events)} races for {SEASON}\n")
        
    except Exception as e:
        logging.error(f"[SEASON FAIL] {SEASON}: {e}")
        print(f"❌ Could not get schedule for {SEASON}: {e}\n")
        continue
    
    # Process each race
    for _, row in race_events.iterrows():
        round_num = row['RoundNumber']
        event_name = row['EventName']
        gp_name = slugify(event_name)
        event_date = row['EventDate']
        
        print(f"  🏁 Round {round_num}: {event_name}")
        
        try:
            # Load session - YOUR EXACT APPROACH
            session = get_session(SEASON, round_num, 'R')
            session.load()
            
            laps = session.laps.reset_index(drop=True)
            
            # YOUR EXACT COLUMN SELECTION
            selected_cols = [
                'Driver', 'Team', 'LapNumber', 'LapTime',
                'Sector1Time', 'Sector2Time', 'Sector3Time',
                'Compound', 'TyreLife', 'Stint',
                'PitInTime', 'PitOutTime', 'TrackStatus',
                'IsAccurate', 'Time'
            ]
            lap_data = laps[selected_cols].copy()

            # Convert time columns to seconds
            for col in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']:
                lap_data[f'{col}Seconds'] = lap_data[col].dt.total_seconds()
            lap_data['LapStartTime'] = lap_data['Time'].dt.total_seconds()

            # Weather Merge (YOUR EXACT CODE)
            try:
                weather = session.weather_data.rename(columns={'Time': 'WeatherTime'})
                weather['WeatherTime'] = weather['WeatherTime'].dt.total_seconds()
                lap_data = pd.merge_asof(
                    lap_data.sort_values('LapStartTime'),
                    weather.sort_values('WeatherTime'),
                    left_on='LapStartTime', right_on='WeatherTime',
                    direction='nearest'
                )
            except Exception:
                logging.warning(f"No weather data for {gp_name}")
                lap_data['Rainfall'] = 0
                lap_data['AirTemp'] = None
                lap_data['Humidity'] = None
                lap_data['WindSpeed'] = None

            # Rain flags
            lap_data['IsWetLap'] = lap_data['Rainfall'] > WET_LAP_THRESHOLD
            lap_data['IsDryLap'] = lap_data['Rainfall'] <= WET_LAP_THRESHOLD
            lap_data['IsWetRace'] = lap_data['IsWetLap'].any()

            # Circuit Metadata
            try:
                circuit_info = session.get_circuit_info()
                lap_data['CircuitName'] = circuit_info.name
                lap_data['CircuitShort'] = circuit_info.location
                lap_data['CircuitCountry'] = circuit_info.country
                lap_data['TrackLengthKM'] = circuit_info.length / 1000 if circuit_info.length else None
                lap_data['AltitudeM'] = circuit_info.altitude
            except Exception:
                event = session.event
                lap_data['CircuitName'] = event.get('OfficialEventName', event_name)
                lap_data['CircuitShort'] = event.get('Location', 'Unknown')
                lap_data['CircuitCountry'] = event.get('Country', 'Unknown')
                lap_data['TrackLengthKM'] = None
                lap_data['AltitudeM'] = None

            lap_data['CircuitType'] = lap_data['CircuitShort'].apply(lambda name: (
                "Street" if isinstance(name, str) and any(x in name.lower() for x in ['monaco', 'baku', 'miami', 'jeddah'])
                else "Hybrid" if isinstance(name, str) and 'marina' in name.lower()
                else "Permanent"
            ))

            # Type & Cleanup
            lap_data['TrackStatus'] = lap_data['TrackStatus'].astype(str)
            lap_data['IsAccurate'] = lap_data['IsAccurate'].astype(bool)
            lap_data['LapNumber'] = lap_data['LapNumber'].fillna(0).astype(int)
            lap_data['TyreLife'] = lap_data['TyreLife'].fillna(0).astype(int)
            lap_data['Stint'] = lap_data['Stint'].fillna(0).astype(int)

            lap_data.dropna(subset=[
                'LapTimeSeconds', 'Sector1TimeSeconds', 'Sector2TimeSeconds',
                'Sector3TimeSeconds', 'Compound'
            ], inplace=True)

            lap_data['Compound'] = lap_data['Compound'].str.upper().map(compound_map).fillna(lap_data['Compound'])

            # Pit Info
            lap_data['PitLap'] = lap_data.apply(
                lambda row: row['LapNumber'] if pd.notna(row['PitInTime']) else None, axis=1
            )
            lap_data['PitDuration'] = (lap_data['PitOutTime'] - lap_data['PitInTime']).dt.total_seconds()

            # Sector Features (YOUR EXACT CODE)
            lap_data['Sector1Pct'] = lap_data['Sector1TimeSeconds'] / lap_data['LapTimeSeconds']
            lap_data['Sector2Pct'] = lap_data['Sector2TimeSeconds'] / lap_data['LapTimeSeconds']
            lap_data['Sector3Pct'] = lap_data['Sector3TimeSeconds'] / lap_data['LapTimeSeconds']
            lap_data['BestSector'] = lap_data[['Sector1TimeSeconds', 'Sector2TimeSeconds', 'Sector3TimeSeconds']].idxmin(axis=1)
            lap_data['BestSector'] = lap_data['BestSector'].str.extract(r'(\d)').astype(int)
            lap_data['IsValidLap'] = (lap_data['TrackStatus'] == 'Green') & lap_data['IsAccurate']

            # Metadata
            lap_data['GrandPrix'] = event_name
            lap_data['GP_Slug'] = gp_name
            lap_data['SeasonYear'] = SEASON
            lap_data['EventDate'] = pd.to_datetime(event_date)

            if 'CarNumber' in laps.columns:
                lap_data['CarNumber'] = laps['CarNumber']

            # Stint Summary (YOUR EXACT CODE)
            stint_summary = lap_data.groupby(['Driver', 'Stint']).agg(
                AvgLapTime=('LapTimeSeconds', 'mean'),
                StintLength=('LapNumber', 'count')
            ).reset_index()
            lap_data = lap_data.merge(stint_summary, on=['Driver', 'Stint'], how='left')

            stint_max_map = lap_data.groupby('Driver')['Stint'].max().to_dict()
            lap_data['StintType'] = lap_data.apply(
                lambda row: "Opening" if row['Stint'] == 1 else
                            "Closing" if row['Stint'] == stint_max_map.get(row['Driver'], 3) else "Mid", axis=1
            )

            # Delta to Driver's Fastest Lap (YOUR EXACT CODE)
            fastest_per_driver = lap_data.groupby("Driver")["LapTimeSeconds"].min().to_dict()
            lap_data['DeltaToFastestLap'] = lap_data.apply(
                lambda row: row["LapTimeSeconds"] - fastest_per_driver.get(row["Driver"], row["LapTimeSeconds"]),
                axis=1
            )

            # Flags (YOUR EXACT CODE)
            lap_data["IsSC"] = lap_data["TrackStatus"].str.contains("4").fillna(False)
            lap_data["IsVSC"] = lap_data["TrackStatus"].str.contains("8").fillna(False)
            lap_data["IsRedFlag"] = lap_data["TrackStatus"].str.contains("16").fillna(False)
            race_max_lap = lap_data["LapNumber"].max()
            lap_data["IsDNF"] = lap_data["LapNumber"] < (race_max_lap - 3)

            # Export CSV
            race_csv = f"{output_dir}/race_summary_{SEASON}_{gp_name}.csv"
            lap_data.to_csv(race_csv, index=False)
            all_races.append(lap_data)
            
            logging.info(f"[EXPORT OK] {gp_name}")
            print(f"    ✅ Exported: {gp_name} ({len(lap_data)} laps)")

        except Exception as e:
            logging.error(f"[PROCESS FAIL] {gp_name}: {e}")
            print(f"    ❌ Failed: {gp_name} — {e}")
    
    # Combined Export for this season
    if all_races:
        combined_df = pd.concat(all_races, ignore_index=True)
        combined_export_path = f"{output_dir}/all_races_combined_{SEASON}.csv"
        combined_df.to_csv(combined_export_path, index=False)
        print(f"\n  🎉 Season {SEASON} complete → {combined_export_path}")
        print(f"     Total laps: {len(combined_df):,}")
        print(f"     Total races: {len(all_races)}\n")
    else:
        print(f"\n  ⚠️  No races processed for {SEASON}\n")

# ========== COMBINE ALL SEASONS ==========
print(f"\n{'='*70}")
print(f"📊 Combining all seasons...")
print(f"{'='*70}\n")

all_season_files = list(Path(output_dir).glob('all_races_combined_*.csv'))
if all_season_files:
    all_dfs = []
    for file in all_season_files:
        df = pd.read_csv(file)
        all_dfs.append(df)
        print(f"  ✅ Loaded: {file.name} ({len(df):,} laps)")
    
    master_df = pd.concat(all_dfs, ignore_index=True)
    master_path = f"{output_dir}/all_races_combined_{START_SEASON}_{END_SEASON}.csv"
    master_df.to_csv(master_path, index=False)
    
    print(f"\n{'='*70}")
    print(f"🎉 COMPLETE!")
    print(f"{'='*70}")
    print(f"\n📊 MASTER FILE:")
    print(f"   {master_path}")
    print(f"   Total records: {len(master_df):,}")
    print(f"   Total races: {master_df['GrandPrix'].nunique()}")
    print(f"   Total drivers: {master_df['Driver'].nunique()}")
    print(f"   Years: {master_df['SeasonYear'].min()}-{master_df['SeasonYear'].max()}")
    
else:
    print("\n⚠️  No season files found to combine.\n")

logger      WARNING 	Failed to load schedule from FastF1 backend!
req            INFO 	No cached data found for season_schedule. Loading data...
_api           INFO 	Fetching season schedule...



🏁 F1 DATA EXTRACTION - USING YOUR WORKING APPROACH
📥 Extracting: 2018-2025
⏱️  Estimated time: 12.0-20.0 hours


🏁 Processing Season 2018



logger      WARNING 	Failed to load schedule from F1 API backend!
logger      WARNING 	Failed to load schedule from Ergast API backend!


❌ Could not get schedule for 2018: Failed to load any schedule data.


🏁 Processing Season 2019



logger      WARNING 	Failed to load schedule from FastF1 backend!
req            INFO 	No cached data found for season_schedule. Loading data...
_api           INFO 	Fetching season schedule...
logger      WARNING 	Failed to load schedule from F1 API backend!
logger      WARNING 	Failed to load schedule from Ergast API backend!


...

❌ Could not get schedule for 2023: Failed to load any schedule data.


🏁 Processing Season 2024



logger      WARNING 	Failed to load schedule from FastF1 backend!
req            INFO 	No cached data found for season_schedule. Loading data...
_api           INFO 	Fetching season schedule...
logger      WARNING 	Failed to load schedule from F1 API backend!
logger      WARNING 	Failed to load schedule from Ergast API backend!


❌ Could not get schedule for 2024: Failed to load any schedule data.


🏁 Processing Season 2025



logger      WARNING 	Failed to load schedule from FastF1 backend!
req            INFO 	No cached data found for season_schedule. Loading data...
_api           INFO 	Fetching season schedule...
logger      WARNING 	Failed to load schedule from F1 API backend!
logger      WARNING 	Failed to load schedule from Ergast API backend!


❌ Could not get schedule for 2025: Failed to load any schedule data.


📊 Combining all seasons...

  ✅ Loaded: all_races_combined_2018_2025.csv (22,473 laps)

🎉 COMPLETE!

📊 MASTER FILE:
   data/processed/all_races_combined_2018_2025.csv
   Total records: 22,473
   Total races: 21
   Total drivers: 26
   Years: 2018-2019

🚀 NEXT STEP:
   streamlit run streamlit_app_enhanced.py

